**Create table on BigQuery**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

# =========================================
# CONFIGURATION
# =========================================

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Labs"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# =========================================
# INITIALIZE CLIENT
# =========================================

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# SCHEMA
# =========================================

schema = [

    bigquery.SchemaField(
        "Lab_ID",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Patient_U_ID",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Lab_Date",
        "DATE"
    ),

    bigquery.SchemaField(
        "Lab_Code",
        "STRING"
    ),

    bigquery.SchemaField(
        "Lab_Group",
        "STRING"
    ),

    bigquery.SchemaField(
        "Lab_Name",
        "STRING"
    ),

    bigquery.SchemaField(
        "Lab_Result",
        "NUMERIC"
    ),

    bigquery.SchemaField(
        "Unit",
        "STRING"
    )
]

# =========================================
# CREATE TABLE
# =========================================

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

table.clustering_fields = [
    "Patient_U_ID",
    "Lab_Code"
]

client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("BigQuery table created successfully")
print(TABLE_REF)
print("=================================")

BigQuery table created successfully
depihealthnux.Depihealthnux_Main.Labs


**Create Table on Postgres**

In [2]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS lab_record_no_seq;

""")

# =========================================
# CREATE TABLE
# =========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS Labs (

    Lab_ID TEXT PRIMARY KEY
    DEFAULT (
        'LI_' ||
        LPAD(
            nextval('lab_record_no_seq')::TEXT,
            4,
            '0'
        )
    ),

    Patient_U_ID TEXT NOT NULL,

    Lab_Date DATE,

    Lab_Code TEXT NOT NULL,

    Lab_Result NUMERIC(12,1),

    Unit TEXT

);
"""

cursor.execute(create_table_query)

# =========================================
# INDEXES
# =========================================

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_labs_patient
ON Labs(Patient_U_ID);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_labs_lab_code
ON Labs(Lab_Code);

""")

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Labs")
print("=================================")

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Labs


**Sync BigQuery to Postgres**

In [3]:
import sys
import pandas as pd
import psycopg2

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ BIGQUERY
# =========================================

query = """

SELECT

    Lab_ID,
    Patient_U_ID,
    Lab_Date,
    Lab_Code,
    Lab_Result,
    Unit

FROM `depihealthnux.Depihealthnux_Main.Labs`

ORDER BY Lab_ID

"""

df = client.query(query).to_dataframe()

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

# =========================================
# CONNECT POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""
DELETE FROM Labs;
""")

if not df.empty:

    execute_values(
        cursor,
        """
        INSERT INTO Labs (

            Lab_ID,
            Patient_U_ID,
            Lab_Date,
            Lab_Code,
            Lab_Result,
            Unit

        )
        VALUES %s
        """,
        df.values.tolist(),
        page_size=1000
    )

# =========================================
# RESET SEQUENCE
# =========================================

cursor.execute("""

SELECT setval(
    'lab_record_no_seq',
    COALESCE(
        (
            SELECT MAX(
                CAST(
                    REPLACE(Lab_ID,'LI_','')
                    AS INTEGER
                )
            )
            FROM Labs
        ),
        1
    ),
    true
);

""")

conn.commit()

print(f"Inserted {len(df)} rows")

cursor.execute("""

SELECT *
FROM Labs
ORDER BY Lab_ID
LIMIT 3

""")

print("\nFirst 3 Rows From PostgreSQL")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

Empty DataFrame
Columns: [Lab_ID, Patient_U_ID, Lab_Date, Lab_Code, Lab_Result, Unit]
Index: []
Rows Retrieved: 0
Inserted 0 rows

First 3 Rows From PostgreSQL


**Sync Postgress to BigQuery**

In [8]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT

    l.Lab_ID,
    l.Patient_U_ID,
    l.Lab_Date,

    l.Lab_Code,

    r.Lab_Group,
    r.Lab_Name,

    l.Lab_Result,
    l.Unit

FROM Labs l

LEFT JOIN Labs_Ref r
    ON l.Lab_Code = r.Lab_Code

ORDER BY l.Lab_ID

"""

df = pd.read_sql(query, conn)

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn.close()

# =========================================
# LOAD TO BIGQUERY
# =========================================

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    df,
    "depihealthnux.Depihealthnux_Main.Labs",
    job_config=job_config
)

job.result()

print(f"Uploaded {len(df)} rows")

# =========================================
# VERIFY
# =========================================

verify_df = client.query("""

SELECT *
FROM `depihealthnux.Depihealthnux_Main.Labs`
LIMIT 3

""").to_dataframe()

print("\nFirst 3 Rows From BigQuery")
print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_8876\1383641763.py:55: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


    lab_id patient_u_id    lab_date lab_code      lab_group lab_name  \
0  LI_0001   PAT_000001  2026-05-20    LC001     DM Profile    HbA1C   
1  LI_0002   PAT_000001  2026-05-20    LC002  Lipid Profile      LDL   
2  LI_0003   PAT_000001  2026-05-20    LC003  Lipid Profile       TG   

   lab_result     unit  
0         6.5    mg/dl  
1       129.0  mmol/ml  
2       300.0  mmol/ml  
Rows Retrieved: 6
Uploaded 6 rows

First 3 Rows From BigQuery
    lab_id patient_u_id    lab_date lab_code      lab_group lab_name  \
0  LI_0006   PAT_000004  2026-05-31    LC001     DM Profile    HbA1C   
1  LI_0001   PAT_000001  2026-05-20    LC001     DM Profile    HbA1C   
2  LI_0002   PAT_000001  2026-05-20    LC002  Lipid Profile      LDL   

   lab_result     unit  
0         5.0    mg/dl  
1         6.5    mg/dl  
2       129.0  mmol/ml  
